# Guided Design Tutorial Notebook

Notebook companion to the [Guided Design Tutorial](../developers-docs/guided-design-tutorial.md).

Use the Markdown tutorial when you want the shortest checked walkthrough. Use this notebook when you want:

- cell-by-cell setup and execution helpers
- inline artifact inspection
- a clearer explanation of where the repo's shipped LLM workflows fit
- a place to branch into your own follow-up checks without leaving the walkthrough

This notebook keeps one deterministic Sc-Zn Zomic-backed spine and adds one bounded companion section for the additive LLM workflows.

## Environment Assumptions

- Launch the notebook from the repo root or from the `materials-discovery/` workspace.
- Run `uv sync --extra dev` before expecting command cells to execute successfully.
- The checked Sc-Zn Zomic path needs a local Java runtime for `mdisc export-zomic` and the Zomic-backed `mdisc generate` path.
- Real/native providers remain optional. The deterministic tutorial path uses checked repo artifacts even if you leave execution preview-only.
- Cells default to **preview mode** so you can inspect commands and artifacts without re-running the whole pipeline by accident.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shlex
import subprocess

ROOT = Path.cwd().resolve()
WORKDIR = ROOT / "materials-discovery" if (ROOT / "materials-discovery").exists() else ROOT
assert (WORKDIR / "configs").exists(), f"Could not find materials-discovery workspace from {ROOT}"

RUN_PIPELINE = False

def run(cmd: list[str], *, execute: bool | None = None, cwd: Path = WORKDIR):
    if execute is None:
        execute = RUN_PIPELINE
    print("$ " + " ".join(shlex.quote(part) for part in cmd))
    if not execute:
        print("(preview only; set RUN_PIPELINE = True to execute this cell)")
        return None
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True, check=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result

def read_json(path: str | Path):
    path = WORKDIR / Path(path)
    return json.loads(path.read_text())

def read_jsonl(path: str | Path, limit: int | None = None):
    path = WORKDIR / Path(path)
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    return rows if limit is None else rows[:limit]

WORKDIR

## 1. Workspace Setup

Run the standard developer sync first. Leave `RUN_PIPELINE = False` if you only want the notebook to preview commands while inspecting the checked repo artifacts.

In [ ]:
run(["uv", "sync", "--extra", "dev"])


## 2. Know the Worked Example

The deterministic walkthrough uses these checked inputs:

- `configs/systems/sc_zn_zomic.yaml`
- `designs/zomic/sc_zn_tsai_bridge.yaml`
- `designs/zomic/sc_zn_tsai_bridge.zomic`
- `data/prototypes/sc_zn_tsai_sczn6.json`

Geometry authority stays on one chain:

`sc_zn_tsai_bridge.zomic` -> `sc_zn_tsai_bridge.raw.json` -> `sc_zn_tsai_bridge.json` -> candidate/report artifacts

The `.zomic` file is the editable source. Everything downstream is derived from it.

In [ ]:
paths = {
    "system_config": WORKDIR / "configs/systems/sc_zn_zomic.yaml",
    "design_yaml": WORKDIR / "designs/zomic/sc_zn_tsai_bridge.yaml",
    "design_source": WORKDIR / "designs/zomic/sc_zn_tsai_bridge.zomic",
    "anchor_prototype": WORKDIR / "data/prototypes/sc_zn_tsai_sczn6.json",
}

{name: str(path.relative_to(WORKDIR)) for name, path in paths.items()}


## 3. Export the Zomic Design

This compiles the Zomic source into the raw labeled geometry export and the orbit-library JSON that `mdisc generate` consumes.

In [ ]:
run([
    "uv", "run", "mdisc", "export-zomic",
    "--design", "designs/zomic/sc_zn_tsai_bridge.yaml",
])

raw = read_json("data/prototypes/generated/sc_zn_tsai_bridge.raw.json")
orbit = read_json("data/prototypes/generated/sc_zn_tsai_bridge.json")

{
    "raw_zomic_file": raw.get("zomic_file"),
    "raw_labeled_points": len(raw.get("labeled_points", [])),
    "raw_segments": len(raw.get("segments", [])),
    "orbit_source_kind": orbit.get("source_kind"),
    "orbit_count": len(orbit.get("orbits", [])),
    "anchor_orbit_summary": orbit.get("anchor_orbit_summary"),
}


## 4. Generate and Screen Candidates

The next two stages create the candidate population and apply the fast screening filters. The checked snapshot teaches the current artifact shape even if you do not rerun the commands.

In [ ]:
run([
    "uv", "run", "mdisc", "generate",
    "--config", "configs/systems/sc_zn_zomic.yaml",
    "--count", "32",
])

candidate = read_jsonl("data/candidates/sc_zn_candidates.jsonl", limit=1)[0]
generate_manifest = read_json("data/manifests/sc_zn_generate_manifest.json")

{
    "candidate_id": candidate.get("candidate_id"),
    "composition": candidate.get("composition"),
    "prototype_key": candidate.get("provenance", {}).get("prototype_key"),
    "prototype_source_kind": candidate.get("provenance", {}).get("prototype_source_kind"),
    "site_count": len(candidate.get("sites", [])),
    "generate_stage": generate_manifest.get("stage"),
}

run([
    "uv", "run", "mdisc", "screen",
    "--config", "configs/systems/sc_zn_zomic.yaml",
])

screen_cal = read_json("data/calibration/sc_zn_screen_calibration.json")
screened = read_jsonl("data/screened/sc_zn_screened.jsonl", limit=1)[0]

{
    "input_count": screen_cal.get("input_count"),
    "passed_count": screen_cal.get("passed_count"),
    "shortlisted_count": screen_cal.get("shortlisted_count"),
    "first_shortlist_rank": screened.get("screen", {}).get("shortlist_rank"),
    "first_energy_proxy_ev_per_atom": screened.get("screen", {}).get("energy_proxy_ev_per_atom"),
    "first_min_distance_proxy": screened.get("screen", {}).get("min_distance_proxy"),
}


## 5. High-Fidelity Validation, Ranking, and Report

These stages tell you whether the batch is actually promising or merely interesting. The current checked Sc-Zn snapshot is useful precisely because it ends in `hold`/`watch` rather than a fake success story.

In [ ]:
run([
    "uv", "run", "mdisc", "hifi-validate",
    "--config", "configs/systems/sc_zn_zomic.yaml",
    "--batch", "all",
])
run([
    "uv", "run", "mdisc", "hifi-rank",
    "--config", "configs/systems/sc_zn_zomic.yaml",
])
run([
    "uv", "run", "mdisc", "report",
    "--config", "configs/systems/sc_zn_zomic.yaml",
])

validated = read_jsonl("data/hifi_validated/sc_zn_all_validated.jsonl", limit=1)[0]
report = read_json("data/reports/sc_zn_report.json")

{
    "validated_candidate_id": validated.get("candidate_id"),
    "geometry_prefilter_pass": validated.get("digital_validation", {}).get("geometry_prefilter_pass"),
    "phonon_imaginary_modes": validated.get("digital_validation", {}).get("phonon_imaginary_modes"),
    "md_stability_score": validated.get("digital_validation", {}).get("md_stability_score"),
    "xrd_confidence": validated.get("digital_validation", {}).get("xrd_confidence"),
    "ranked_count": report.get("ranked_count"),
    "reported_count": report.get("reported_count"),
    "release_gate": report.get("release_gate"),
    "top_recommendation": (report.get("summary") or {}).get("recommendation"),
}


## 6. How to Read the Current Evidence

The checked artifact snapshot is dated, so treat exact counts as a point-in-time reading, not a timeless promise.

What matters operationally:

- `shortlisted_count` tells you how many candidates survived fast screening strongly enough to justify expensive validation.
- `geometry_prefilter_pass` is the first cheap crowding gate before the expensive phonon/MD path.
- `phonon_imaginary_modes` and `md_stability_score` are stability warnings, not decorative metrics.
- `recommendation`, `priority`, `risk_flags`, `stability_probability`, and `release_gate` are the operator-facing summary of whether the run is ready for serious follow-up.

For the current checked Sc-Zn run, the honest reading is still: the batch is useful, but it remains a watchlist rather than a promotion-ready set.

## 7. Where the LLM Workflows Fit

The deterministic Sc-Zn path above is still the clearest first walkthrough. The repo's LLM workflows are additive around that spine.

### Same-System Companion Lane

Use these configs when you want the same Sc-Zn family but a different workflow lane:

- `configs/systems/sc_zn_llm_mock.yaml` — fixture-backed LLM generation/evaluation lane
- `configs/systems/sc_zn_llm_local.yaml` — local OpenAI-compatible lane with general-purpose generation and specialized evaluation

The key rule is unchanged: `.zomic` stays the geometry authority for the checked design path, and LLM outputs are additive proposals or assessments, not a reason to stop reading the deterministic evidence.

### Other Shipped Workflow Families

- **Additive generation and assessment:** `llm-generate`, `llm-evaluate`
- **Campaign governance:** `llm-suggest`, `llm-approve`, `llm-launch`, `llm-replay`, `llm-compare`
- **Serving and checkpoints:** `llm-serving-benchmark`, `llm-register-checkpoint`, `llm-list-checkpoints`, `llm-promote-checkpoint`, `llm-retire-checkpoint`
- **Translation and external benchmarking:** `llm-translate`, `llm-translate-inspect`, `llm-translated-benchmark-freeze`, `llm-translated-benchmark-inspect`, `llm-register-external-target`, `llm-inspect-external-target`, `llm-smoke-external-target`, `llm-external-benchmark`, `llm-inspect-external-benchmark`

Reference docs:

- [LLM Integration](../developers-docs/llm-integration.md)
- [Operator Runbook](../RUNBOOK.md)
- [LLM Translation Runbook](../developers-docs/llm-translation-runbook.md)
- [Translated Benchmark Runbook](../developers-docs/llm-translated-benchmark-runbook.md)
- [External Target Runbook](../developers-docs/llm-external-target-runbook.md)
- [External Benchmark Runbook](../developers-docs/llm-external-benchmark-runbook.md)

In [ ]:
run([
    "uv", "run", "mdisc", "llm-generate",
    "--config", "configs/systems/sc_zn_llm_mock.yaml",
    "--count", "5",
    "--out", "data/candidates/sc_zn_llm_candidates.jsonl",
])
run([
    "uv", "run", "mdisc", "llm-evaluate",
    "--config", "configs/systems/sc_zn_llm_mock.yaml",
    "--batch", "all",
])

llm_artifacts = {
    "llm_candidate_output": str((WORKDIR / "data/candidates/sc_zn_llm_candidates.jsonl").relative_to(WORKDIR)),
    "llm_generate_manifest": str((WORKDIR / "data/manifests/sc_zn_llm_generate_manifest.json").relative_to(WORKDIR)),
    "llm_evaluated_output": str((WORKDIR / "data/llm_evaluated/sc_zn_all_llm_evaluated.jsonl").relative_to(WORKDIR)),
    "llm_evaluate_manifest": str((WORKDIR / "data/manifests/sc_zn_all_llm_evaluate_manifest.json").relative_to(WORKDIR)),
}
llm_artifacts


## 8. Visualize and Iterate in vZome

Keep `designs/zomic/sc_zn_tsai_bridge.zomic` as the editable motif source.

1. Open that `.zomic` file in the vZome desktop app's Zomic editor.
2. Run the script to inspect the motif directly.
3. After edits, rerun `mdisc export-zomic` before trusting any downstream candidates again.
4. Compare the next report's `recommendation`, `risk_flags`, and `release_gate` against the previous run.

That loop keeps geometry, pipeline evidence, and any later LLM companion work attached to one auditable artifact chain.